# The math of classification: sigmoid, softmax and gradients

A lesson from the Python and AI School, by Dr. Behrooz Taheri

## From a score to a probability
A linear classifier first builds a **score**: $z=w\cdot x+b$. The score can be any number; the sigmoid turns it into a probability:

$$p=\sigma(z)=\frac{1}{1+e^{-z}}$$

The meaning of z is the **log-odds**. If $p=0.8$, the odds are $0.8/0.2=4$ and $z=\ln4=1.386$.

| z | −3 | −1 | 0 | 1 | 3 |
|---|---|---|---|---|---|
| p | 0.047 | 0.269 | 0.5 | 0.731 | 0.953 |
| odds $p/(1-p)$ | 0.05 | 0.37 | 1 | 2.72 | 20.1 |

## The loss
From the likelihood lesson: the binary cross-entropy

$$L=-\big[y\ln p+(1-y)\ln(1-p)\big]$$

> [Log loss: if the model is confidently wrong, the penalty becomes very large.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/logit_loss.png)

*Log loss: if the model is confidently wrong, the penalty becomes very large.*

## The gradient: a beautiful result
With the chain rule $\dfrac{\partial L}{\partial z}=\dfrac{\partial L}{\partial p}\cdot\dfrac{\partial p}{\partial z}$:

| Part | Value |
|---|---|
| $\partial L/\partial p$ | $-\dfrac{y}{p}+\dfrac{1-y}{1-p}=\dfrac{p-y}{p(1-p)}$ |
| $\partial p/\partial z$ | $p(1-p)$ |
| Product | $p-y$ |

$$\frac{\partial L}{\partial z}=p-y\qquad\frac{\partial L}{\partial w}=(p-y)\,x\qquad\frac{\partial L}{\partial b}=p-y$$

The gradient is just "prediction minus truth" times the input; exactly the form we had in linear regression.

## Worked example: one training step
$x=(2,1)$, $y=1$, $w=(0.5,-1)$, $b=0.5$, $\eta=0.1$:

| Step | Calculation | Value |
|---|---|---|
| Score z | $0.5\times2-1\times1+0.5$ | 0.5 |
| Probability p | $\sigma(0.5)$ | 0.622 |
| Loss | $-\ln0.622$ | 0.474 |
| $p-y$ | $0.622-1$ | −0.378 |
| $\partial L/\partial w$ | $-0.378\times(2,1)$ | $(-0.755,\ -0.378)$ |
| New w | $(0.5,-1)-0.1(-0.755,-0.378)$ | $(0.576,\ -0.962)$ |
| New b | $0.5+0.0378$ | 0.538 |
| New p | $\sigma(0.727)$ | 0.674 |
| New loss | $-\ln0.674$ | 0.394 |

The loss went from 0.474 to 0.394.

## More than two classes: softmax
For K classes we have K scores, and softmax turns them into a probability distribution:

$$p_k=\frac{e^{z_k}}{\sum_j e^{z_j}}$$

> Open the interactive lab for this part on the website: https://behrooztaheri.github.io/learn-python-ai/#/l/math-logit

Example with $z=(2,\ 1,\ 0.1)$:

| Class | $z_k$ | $e^{z_k}$ | $p_k$ |
|---|---|---|---|
| 1 | 2 | 7.389 | 0.659 |
| 2 | 1 | 2.718 | 0.242 |
| 3 | 0.1 | 1.105 | 0.099 |
| Sum | | 11.212 | 1 |

If the true class is class 1, the loss is $-\ln0.659=0.417$ and the gradient again has the same simple form:

$$\frac{\partial L}{\partial z}=p-y=(0.659-1,\ 0.242,\ 0.099)=(-0.341,\ 0.242,\ 0.099)$$

## Properties of softmax
| Property | Explanation |
|---|---|
| Sums to 1 | A valid probability distribution |
| Keeps the order | The largest score gets the largest probability |
| Depends on differences | Adding a constant to every score changes nothing |
| Temperature T | $z/T$: low T = more decisive, high T = more uniform ("creativity" in language models) |
| K = 2 | It is the sigmoid |

> In code never compute $e^{z}$ directly: for $z=1000$ it overflows. Subtract the largest score first (`z - z.max()`). Libraries compute softmax and the log together in a stable way (log-sum-exp).

## The decision boundary
The model picks class 1 when $p>0.5$, that is $z>0$, that is $w\cdot x+b>0$. So the decision boundary is **a line** (or hyperplane) perpendicular to w, which is why logistic regression is a **linear** classifier.



In [ ]:
import numpy as np

sigmoid = lambda z: 1 / (1 + np.exp(-z))

# One training step of logistic regression, as in the lesson
x = np.array([2.0, 1.0]); y = 1.0
w = np.array([0.5, -1.0]); b = 0.5; eta = 0.1
for step in range(3):
    z = w @ x + b; p = sigmoid(z)
    loss = -(y * np.log(p) + (1 - y) * np.log(1 - p))
    print(f"step {step}: z = {z:.3f}  p = {p:.3f}  loss = {loss:.3f}  grad_w = {((p - y) * x).round(3)}")
    w = w - eta * (p - y) * x
    b = b - eta * (p - y)

# Check that dL/dz = p - y with a numerical derivative
L = lambda z: -np.log(sigmoid(z))               # y = 1
h = 1e-6
print("numeric dL/dz at z = 0.5:", round((L(0.5 + h) - L(0.5 - h)) / (2 * h), 4), "   p - y =", round(sigmoid(0.5) - 1, 4))

def softmax(z, T=1.0):
    z = np.asarray(z, dtype=float) / T
    e = np.exp(z - z.max())                      # subtract the max: numerically stable
    return e / e.sum()

z = np.array([2.0, 1.0, 0.1])
p = softmax(z)
print("softmax:", p.round(3), "  sum =", p.sum())
print("loss if class 1 is true:", round(-np.log(p[0]), 3), "  gradient p - y:", (p - np.array([1, 0, 0])).round(3))
for T in [0.2, 1, 5]:
    print(f"T = {T}:", softmax(z, T).round(3))
print("huge scores still work:", softmax([1000, 999, 0]).round(3))

# Train logistic regression from scratch on two clouds of points
rng = np.random.default_rng(0)
X = np.r_[rng.normal([-1, -1], 1, (100, 2)), rng.normal([1.5, 1], 1, (100, 2))]
t = np.r_[np.zeros(100), np.ones(100)]
w = np.zeros(2); b = 0.0
for epoch in range(300):
    p = sigmoid(X @ w + b)
    w -= 0.1 * X.T @ (p - t) / 200
    b -= 0.1 * np.mean(p - t)
print("w =", w.round(3), " b =", round(b, 3), " accuracy =", np.mean((sigmoid(X @ w + b) > 0.5) == t))

### Exercise

Compute the softmax of $z=(1,2,3)$ by hand. If the true class is class 3, what is the loss?

In [ ]:
# Your answer here

### Exercise

In the lab move the temperature to 0.1 and then to 5. What changes?

In [ ]:
# Your answer here